# VeriFact — Fake News Detector
## Notebook 3: NLP Enrichment Layer
**Goal:** Add sentiment analysis, tone scoring, and Named Entity Recognition (NER) to enrich predictions.  
These outputs feed directly into the Streamlit app UI.

---

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = '/content/drive/MyDrive/VeriFact/data/WELFake_Cleaned.csv'
MODEL_PATH = '/content/drive/MyDrive/VeriFact/models/'

In [ ]:
!pip install vaderSentiment langdetect --quiet
!python -m spacy download en_core_web_sm --quiet
print("Done.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import re
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import spacy
from langdetect import detect, LangDetectException

# Load spaCy model
nlp = spacy.load('en_core_web_sm', disable=['parser'])
# VADER sentiment analyzer
vader = SentimentIntensityAnalyzer()

FAKE_COLOR = '#E24B4A'
REAL_COLOR = '#1D9E75'

print("NLP libraries loaded.")

## Step 1 — Core NLP Functions

These functions will be copied into `utils/nlp_utils.py` for the Streamlit app.

In [ ]:
# ── 1. Language Detection ────────────────────────────────────────────
def detect_language(text):
    """
    Returns ISO 639-1 language code ('en', 'hi', 'fr', etc.)
    Returns 'unknown' if detection fails.
    """
    try:
        return detect(str(text)[:500])  # only need first 500 chars
    except LangDetectException:
        return 'unknown'


# ── 2. Sentiment Analysis (VADER) ────────────────────────────────────
def get_sentiment(text):
    """
    Returns a dict with:
      - compound: overall sentiment score (-1 to +1)
      - label:    'Positive', 'Negative', or 'Neutral'
      - scores:   full VADER breakdown
    """
    scores = vader.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    return {
        'compound': round(compound, 4),
        'label':    label,
        'positive': round(scores['pos'], 4),
        'negative': round(scores['neg'], 4),
        'neutral':  round(scores['neu'], 4)
    }


# ── 3. Sensationalism / Tone Score ───────────────────────────────────
def get_tone_score(text):
    """
    Computes a 0–100 sensationalism score based on:
      - Exclamation marks (!)
      - Question marks (?)
      - All-caps words
      - Sensationalist keywords
    Higher score = more sensationalist = more likely fake
    """
    text_str = str(text)
    words    = text_str.split()
    total    = max(len(words), 1)

    excl_score  = min(text_str.count('!') * 5, 30)
    ques_score  = min(text_str.count('?') * 3, 15)
    caps_score  = min(sum(1 for w in words if w.isupper() and len(w) > 2) / total * 100, 25)

    SENSATIONAL_WORDS = [
        'breaking', 'shocking', 'unbelievable', 'exposed', 'secret',
        'banned', 'conspiracy', 'hoax', 'fake', 'fraud', 'urgent',
        'alert', 'confirmed', 'truth', 'mainstream media', 'they dont want',
        'share before', 'before its deleted', 'wake up'
    ]
    text_lower   = text_str.lower()
    keyword_hits = sum(1 for kw in SENSATIONAL_WORDS if kw in text_lower)
    keyword_score = min(keyword_hits * 5, 30)

    total_score = excl_score + ques_score + caps_score + keyword_score
    return min(round(total_score, 1), 100)


# ── 4. Named Entity Recognition ──────────────────────────────────────
def extract_entities(text, max_chars=2000):
    """
    Extracts named entities using spaCy.
    Returns dict of {entity_type: [list of entities]}
    """
    doc = nlp(str(text)[:max_chars])
    entities = {}
    for ent in doc.ents:
        etype = ent.label_
        if etype not in entities:
            entities[etype] = []
        if ent.text not in entities[etype]:
            entities[etype].append(ent.text)
    return entities


# ── 5. Full NLP Analysis Pipeline ────────────────────────────────────
def analyze_article(text):
    """
    Runs all NLP enrichment on a single article.
    Returns a comprehensive dict for use in Streamlit app.
    """
    lang       = detect_language(text)
    sentiment  = get_sentiment(text)
    tone_score = get_tone_score(text)
    entities   = extract_entities(text)

    # Key entity types for display
    people  = entities.get('PERSON', [])[:5]
    orgs    = entities.get('ORG', [])[:5]
    places  = entities.get('GPE', [])[:5] + entities.get('LOC', [])[:3]
    dates   = entities.get('DATE', [])[:3]

    return {
        'language':    lang,
        'is_english':  lang == 'en',
        'sentiment':   sentiment,
        'tone_score':  tone_score,
        'entities': {
            'people':  people,
            'orgs':    orgs,
            'places':  places,
            'dates':   dates
        }
    }


print("All NLP functions defined.")

## Step 2 — Test on Sample Articles

In [ ]:
# Sample 1: Fake-style article
fake_article = """
BREAKING!! Scientists are HIDING the truth about vaccines!! The government 
doesn't want you to know this secret. Share before they delete it! Obama and 
Hillary Clinton are involved in a massive conspiracy. WAKE UP AMERICA!!
"""

# Sample 2: Real-style article
real_article = """
The Reserve Bank of India raised its benchmark interest rate by 25 basis points 
on Friday, according to Governor Shaktikanta Das. The Monetary Policy Committee 
voted 4-2 in favour of the increase, citing persistent inflation concerns.
New Delhi-based economists broadly expected the move following last month's data.
"""

print("=" * 60)
print("FAKE ARTICLE ANALYSIS:")
fake_result = analyze_article(fake_article)
print(f"  Language      : {fake_result['language']}")
print(f"  Sentiment     : {fake_result['sentiment']['label']} (compound: {fake_result['sentiment']['compound']})")
print(f"  Tone Score    : {fake_result['tone_score']} / 100")
print(f"  People found  : {fake_result['entities']['people']}")
print(f"  Orgs found    : {fake_result['entities']['orgs']}")

print("\n" + "=" * 60)
print("REAL ARTICLE ANALYSIS:")
real_result = analyze_article(real_article)
print(f"  Language      : {real_result['language']}")
print(f"  Sentiment     : {real_result['sentiment']['label']} (compound: {real_result['sentiment']['compound']})")
print(f"  Tone Score    : {real_result['tone_score']} / 100")
print(f"  People found  : {real_result['entities']['people']}")
print(f"  Orgs found    : {real_result['entities']['orgs']}")
print(f"  Places found  : {real_result['entities']['places']}")

## Step 3 — Validate on a Sample of the Dataset

In [ ]:
# Sample 500 articles for validation (full run takes too long in notebook)
df = pd.read_csv(DATA_PATH).dropna(subset=['content'])
sample = df.sample(500, random_state=42).copy()

print("Running NLP analysis on 500 sample articles...")
sample['sentiment_compound'] = sample['content'].apply(lambda x: get_sentiment(x)['compound'])
sample['tone_score']         = sample['content'].apply(get_tone_score)
sample['label_name']         = sample['label'].map({0: 'Fake', 1: 'Real'})
print("Done.")

# Stats
print("\nAverage sentiment (compound) by label:")
print(sample.groupby('label_name')['sentiment_compound'].mean())

print("\nAverage tone score by label:")
print(sample.groupby('label_name')['tone_score'].mean())

## Step 4 — Visualization: Sentiment & Tone by Label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NLP Features — Fake vs Real News', fontsize=15, fontweight='bold')

# Sentiment distribution
for label, color in [('Fake', FAKE_COLOR), ('Real', REAL_COLOR)]:
    data = sample[sample['label_name'] == label]['sentiment_compound']
    axes[0].hist(data, bins=30, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Sentiment Score Distribution')
axes[0].set_xlabel('VADER Compound Score (-1 to +1)')
axes[0].set_ylabel('Density')
axes[0].axvline(0, color='gray', linestyle='--', linewidth=1)
axes[0].legend()

# Tone score distribution
for label, color in [('Fake', FAKE_COLOR), ('Real', REAL_COLOR)]:
    data = sample[sample['label_name'] == label]['tone_score']
    axes[1].hist(data, bins=20, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Sensationalism Tone Score Distribution')
axes[1].set_xlabel('Tone Score (0 = calm, 100 = sensational)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/VeriFact/assets/09_nlp_features.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Export NLP Utils as Python File

Save `nlp_utils.py` to Drive — this file is imported directly by the Streamlit app.

In [ ]:
nlp_utils_code = '''
import re
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from langdetect import detect, LangDetectException

nlp   = spacy.load("en_core_web_sm", disable=["parser"])
vader = SentimentIntensityAnalyzer()

def detect_language(text):
    try:
        return detect(str(text)[:500])
    except LangDetectException:
        return "unknown"

def get_sentiment(text):
    scores   = vader.polarity_scores(str(text))
    compound = scores["compound"]
    label    = "Positive" if compound >= 0.05 else ("Negative" if compound <= -0.05 else "Neutral")
    return {"compound": round(compound, 4), "label": label,
            "positive": round(scores["pos"], 4),
            "negative": round(scores["neg"], 4),
            "neutral":  round(scores["neu"], 4)}

def get_tone_score(text):
    text_str = str(text)
    words    = text_str.split()
    total    = max(len(words), 1)
    excl_score    = min(text_str.count("!") * 5, 30)
    ques_score    = min(text_str.count("?") * 3, 15)
    caps_score    = min(sum(1 for w in words if w.isupper() and len(w) > 2) / total * 100, 25)
    SENSATIONAL_WORDS = ["breaking","shocking","unbelievable","exposed","secret",
        "banned","conspiracy","hoax","fraud","urgent","alert","confirmed",
        "truth","mainstream media","share before","before its deleted","wake up"]
    keyword_score = min(sum(1 for kw in SENSATIONAL_WORDS if kw in text_str.lower()) * 5, 30)
    return min(round(excl_score + ques_score + caps_score + keyword_score, 1), 100)

def extract_entities(text, max_chars=2000):
    doc = nlp(str(text)[:max_chars])
    entities = {}
    for ent in doc.ents:
        entities.setdefault(ent.label_, [])
        if ent.text not in entities[ent.label_]:
            entities[ent.label_].append(ent.text)
    return entities

def analyze_article(text):
    lang      = detect_language(text)
    sentiment = get_sentiment(text)
    tone      = get_tone_score(text)
    entities  = extract_entities(text)
    return {
        "language":   lang,
        "is_english": lang == "en",
        "sentiment":  sentiment,
        "tone_score": tone,
        "entities":   {
            "people": entities.get("PERSON", [])[:5],
            "orgs":   entities.get("ORG",    [])[:5],
            "places": entities.get("GPE",    [])[:5],
            "dates":  entities.get("DATE",   [])[:3]
        }
    }
'''

save_path = '/content/drive/MyDrive/VeriFact/utils/nlp_utils.py'
import os
os.makedirs('/content/drive/MyDrive/VeriFact/utils/', exist_ok=True)
with open(save_path, 'w') as f:
    f.write(nlp_utils_code.strip())

print(f"nlp_utils.py saved to: {save_path}")

## Summary

| NLP Feature | Method | Signal for Fake News |
|---|---|---|
| Language detection | langdetect | Routes non-English to Gemini |
| Sentiment analysis | VADER | Fake news tends to be more negative/extreme |
| Tone score | Rule-based | High exclamation, caps, keywords = sensational |
| NER | spaCy | Extracts who/what/where for student context |

**Next step →** Notebook 4: Gemini API Integration